# Direct fine-tuning — MoViNet-A0 shoplifting detector

Fully runnable Google Colab pipeline with automatic data download, robust temporal sampling, GPU training, validation, checkpointing, threshold selection, visual evaluation, and inference benchmarking.

In [ ]:
# Colab setup
!nvidia-smi -L
!pip -q install -U "transformers>=4.45" "accelerate>=0.34" "huggingface_hub>=0.25" "decord>=0.6.0" "scikit-learn>=1.4" "pandas>=2.0" "matplotlib>=3.8" "seaborn>=0.13" "tqdm>=4.66"


In [ ]:
from pathlib import Path
import json, random, time
import numpy as np, pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support, roc_auc_score, average_precision_score, confusion_matrix, classification_report, roc_curve, precision_recall_curve
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm
import decord
decord.bridge.set_bridge("native")

assert torch.cuda.is_available(), "Enable a Colab GPU first."
DEVICE=torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.benchmark=True
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

ROOT=Path("/content/shoplifting_project"); DATA=ROOT/"data"; CKPT=ROOT/"checkpoints"; FIG=ROOT/"figures"
for p in (DATA,CKPT,FIG): p.mkdir(parents=True,exist_ok=True)

UCF_REPO="jinmang2/ucf_crime"
TRAIN_LIST="UCF_Crimes-Train-Test-Split/Action_Recognition_splits/train_001.txt"
VAL_LIST="UCF_Crimes-Train-Test-Split/Action_Recognition_splits/test_001.txt"
MODEL_ID="kfkas/movinet-a0-stream-pytorch"
LABELS=["normal","shoplifting"]; LABEL2ID={"normal":0,"shoplifting":1}
NUM_FRAMES=16; IMAGE_SIZE=172; EPOCHS=5; LR=2e-4; WEIGHT_DECAY=1e-4
VAL_CLIPS=3; MAX_TRAIN_VIDEOS=0; MAX_VAL_VIDEOS=0
VRAM=torch.cuda.get_device_properties(0).total_memory/2**30
BATCH_SIZE=8 if VRAM>=20 else 4 if VRAM>=10 else 2
NUM_WORKERS=2
print(torch.cuda.get_device_name(0), f"{VRAM:.1f} GB VRAM; batch={BATCH_SIZE}")


In [ ]:
def read_split(path):
    local=hf_hub_download(repo_id=UCF_REPO,filename=path,repo_type="dataset")
    return [s.strip() for s in Path(local).read_text(errors="ignore").splitlines() if s.strip().lower().endswith(".mp4")]

def choose_balanced(path, seed):
    items=read_split(path)
    pos=[(x,1) for x in items if x.startswith("Shoplifting/")]
    neg=[(x,0) for x in items if "Normal_Videos" in x]
    rng=random.Random(seed); rng.shuffle(pos); rng.shuffle(neg)
    n=min(len(pos),len(neg))
    return pos[:n]+neg[:n]

train_items=choose_balanced(TRAIN_LIST,SEED)
val_items=choose_balanced(VAL_LIST,SEED+1)
if MAX_TRAIN_VIDEOS: train_items=train_items[:MAX_TRAIN_VIDEOS]
if MAX_VAL_VIDEOS: val_items=val_items[:MAX_VAL_VIDEOS]
print("selected",len(train_items),"train and",len(val_items),"validation videos")

def download_items(items, split):
    rows=[]
    for rel,label in tqdm(items,desc="download "+split):
        p=hf_hub_download(repo_id=UCF_REPO,filename=rel,repo_type="dataset",local_dir=str(DATA/split))
        rows.append({"path":str(p),"label":label,"rel":rel})
    return rows

train_rows=download_items(train_items,"train")
val_rows=download_items(val_items,"val")
pd.DataFrame(train_rows+val_rows).to_csv(ROOT/"manifest.csv",index=False)


In [ ]:
def inspect_rows(rows):
    good=[]
    for r in tqdm(rows,desc="checking videos"):
        try:
            n=len(decord.VideoReader(r["path"],ctx=decord.cpu(0)))
            if n>=2: good.append({**r,"frames":n})
        except Exception as e:
            pass
    return good

train_rows=inspect_rows(train_rows); val_rows=inspect_rows(val_rows)
assert train_rows and val_rows, "No readable videos remain."
print("readable:",len(train_rows),len(val_rows))

def sample_indices(total, random_start=True):
    if total<=NUM_FRAMES:
        return np.linspace(0,total-1,NUM_FRAMES).round().astype(np.int64)
    start=random.randint(0,total-NUM_FRAMES) if random_start else 0
    return np.arange(start,start+NUM_FRAMES,dtype=np.int64)

def decode(path, indices):
    vr=decord.VideoReader(path,ctx=decord.cpu(0))
    return vr.get_batch(indices).asnumpy()

def preprocess(frames, augment=False):
    x=torch.from_numpy(frames).permute(0,3,1,2).float()/255.0
    x=F.interpolate(x,size=(IMAGE_SIZE,IMAGE_SIZE),mode="bilinear",align_corners=False)
    if augment and random.random()<0.5: x=torch.flip(x,[-1])
    return x

def val_starts(total):
    if total<=NUM_FRAMES: return [0]
    return np.linspace(0,total-NUM_FRAMES,VAL_CLIPS).round().astype(int).tolist()

class VideoDS(Dataset):
    def __init__(self,rows,train=False): self.rows=rows; self.train=train
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        if self.train:
            x=preprocess(decode(r["path"],sample_indices(r["frames"],True)),True)
        else:
            x=torch.stack([preprocess(decode(r["path"],np.arange(s,s+NUM_FRAMES))) for s in val_starts(r["frames"])])
        return x,torch.tensor(r["label"],dtype=torch.long),r["rel"]

train_loader=DataLoader(VideoDS(train_rows,True),batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
val_loader=DataLoader(VideoDS(val_rows,False),batch_size=max(1,BATCH_SIZE//2),shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,persistent_workers=True)
x,y,_=next(iter(train_loader)); print("train batch",tuple(x.shape))


In [ ]:
import matplotlib.pyplot as plt
r=random.choice(train_rows)
frames=decode(r["path"],sample_indices(r["frames"],False))
fig,axs=plt.subplots(2,8,figsize=(18,5))
for ax,fr in zip(axs.ravel(),frames):
    ax.imshow(fr); ax.axis("off")
fig.suptitle(f"label={LABELS[r['label']]} | {r['rel']}")
plt.tight_layout(); plt.show()


In [ ]:
from transformers import AutoModelForVideoClassification
model = AutoModelForVideoClassification.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    num_labels=2,
    ignore_mismatched_sizes=True,
    label2id=LABEL2ID,
    id2label={0: "normal", 1: "shoplifting"},
).to(DEVICE)
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")


In [ ]:
def logits(out):
    return out.logits if hasattr(out,"logits") else out

def evaluate(model,loader):
    model.eval(); ys=[]; ps=[]; probs=[]
    with torch.inference_mode():
        for x,y,_ in tqdm(loader,desc="validation",leave=False):
            b,k,t,c,h,w=x.shape
            x=x.reshape(b*k,t,c,h,w).to(DEVICE,non_blocking=True)
            with torch.autocast("cuda",dtype=torch.float16):
                z=logits(model(pixel_values=x))
            z=z.float().reshape(b,k,2).mean(1)
            p=z.softmax(-1)[:,1]
            ys.extend(y.tolist()); ps.extend(z.argmax(-1).cpu().tolist()); probs.extend(p.cpu().tolist())
    pr,re,f1,_=precision_recall_fscore_support(ys,ps,average="binary",zero_division=0)
    return {"accuracy":accuracy_score(ys,ps),"balanced_accuracy":balanced_accuracy_score(ys,ps),
            "precision":pr,"recall":re,"f1":f1,
            "roc_auc":roc_auc_score(ys,probs),"average_precision":average_precision_score(ys,probs),
            "y":ys,"p":ps,"prob":probs}

optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
scaler=torch.amp.GradScaler("cuda")
best_path=CKPT/"movinet_direct_best.pt"; best_f1=-1; history=[]

for epoch in range(1,EPOCHS+1):
    model.train(); total=0.0
    for x,y,_ in tqdm(train_loader,desc=f"epoch {epoch}/{EPOCHS}"):
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=torch.float16):
            loss=F.cross_entropy(logits(model(pixel_values=x)),y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optimizer); scaler.update(); total+=loss.item()
    scheduler.step()
    m=evaluate(model,val_loader)
    row={"epoch":epoch,"train_loss":total/len(train_loader),"lr":optimizer.param_groups[0]["lr"],
         **{k:v for k,v in m.items() if k not in ("y","p","prob")}}
    history.append(row); print(row)
    if m["f1"]>best_f1:
        best_f1=m["f1"]
        torch.save({"model_state_dict":model.state_dict(),"model_id":MODEL_ID,"num_frames":NUM_FRAMES,
                    "image_size":IMAGE_SIZE,"labels":LABELS,"metrics":row},best_path)
pd.DataFrame(history).to_csv(ROOT/"direct_history.csv",index=False)
print("best F1:",best_f1)


In [ ]:
bundle=torch.load(best_path,map_location=DEVICE)
model.load_state_dict(bundle["model_state_dict"])
final=evaluate(model,val_loader)
metrics={k:float(v) for k,v in final.items() if k not in ("y","p","prob")}
print(json.dumps(metrics,indent=2))
print(classification_report(final["y"],final["p"],target_names=LABELS,digits=4))
(ROOT/"final_metrics.json").write_text(json.dumps(metrics,indent=2))

cm=confusion_matrix(final["y"],final["p"])
import seaborn as sns
plt.figure(figsize=(5,4)); sns.heatmap(cm,annot=True,fmt="d",xticklabels=LABELS,yticklabels=LABELS)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix"); plt.tight_layout(); plt.savefig(FIG/"confusion_matrix.png",dpi=160); plt.show()

fpr,tpr,_=roc_curve(final["y"],final["prob"]); prec,rec,_=precision_recall_curve(final["y"],final["prob"])
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(fpr,tpr); ax[0].plot([0,1],[0,1],"--"); ax[0].set_title(f"ROC AUC={final['roc_auc']:.3f}")
ax[1].plot(rec,prec); ax[1].set_title(f"PR AP={final['average_precision']:.3f}")
plt.tight_layout(); plt.savefig(FIG/"roc_pr.png",dpi=160); plt.show()

h=pd.DataFrame(history)
plt.figure(figsize=(8,4)); plt.plot(h.epoch,h.train_loss,marker="o"); plt.title("Training loss"); plt.xlabel("Epoch"); plt.grid(alpha=.2); plt.tight_layout(); plt.savefig(FIG/"training_loss.png",dpi=160); plt.show()


In [ ]:
thresholds=np.linspace(0.05,0.95,181)
f1s=[]
for th in thresholds:
    pp=(np.asarray(final["prob"])>=th).astype(int)
    f1s.append(precision_recall_fscore_support(final["y"],pp,average="binary",zero_division=0)[2])
threshold=float(thresholds[int(np.argmax(f1s))])
print("validation F1-optimal threshold:",threshold)
torch.save({"model_state_dict":model.state_dict(),"model_id":MODEL_ID,"labels":LABELS,
            "num_frames":NUM_FRAMES,"image_size":IMAGE_SIZE,"threshold":threshold,
            "validation_metrics":metrics},CKPT/"movinet_direct_deployment.pt")

def predict_video(path):
    total=len(decord.VideoReader(path,ctx=decord.cpu(0)))
    clips=[preprocess(decode(path,np.arange(s,s+NUM_FRAMES))) for s in val_starts(total)]
    x=torch.stack(clips).to(DEVICE)
    with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16):
        z=logits(model(pixel_values=x)).float().mean(0)
    prob=float(z.softmax(-1)[1].cpu())
    return {"label":"shoplifting" if prob>=threshold else "normal","shoplifting_probability":prob,"threshold":threshold}
print(predict_video(train_rows[0]["path"]))


In [ ]:
# GPU inference benchmark
model.eval()
bx=next(iter(train_loader))[0].to(DEVICE)
for _ in range(5):
    with torch.inference_mode(),torch.autocast("cuda",dtype=torch.float16): _=logits(model(pixel_values=bx))
torch.cuda.synchronize()
t0=time.perf_counter(); n=30
with torch.inference_mode():
    for _ in range(n):
        with torch.autocast("cuda",dtype=torch.float16): _=logits(model(pixel_values=bx))
torch.cuda.synchronize()
elapsed=time.perf_counter()-t0
print(f"batch latency: {elapsed/n*1000:.2f} ms")
print(f"per-clip latency: {elapsed/n/len(bx)*1000:.2f} ms")
print(f"throughput: {n*len(bx)/elapsed:.2f} clips/s")
